In [5]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

2.11.0+cu130 True Tesla T4


In [6]:
# start clean so old and new photos can't mix
!rm -rf raw
!unzip -q vision_data.zip
!ls raw/pen | wc -l
!ls raw/cube | wc -l

29
25


In [8]:
from torchvision import datasets, transforms

# resize every photo to 224x224 and turn it into numbers
tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
# labels come from the folder names
data = datasets.ImageFolder("raw", transform=tfm)
print(data.classes, data.class_to_idx, len(data))

['cube', 'pen'] {'cube': 0, 'pen': 1} 54


In [9]:
from torch.utils.data import random_split

# hold out 20% of the photos for testing
n_test = int(0.2 * len(data))
n_train = len(data) - n_test
# fixed seed so the split is the same every run
g = torch.Generator().manual_seed(42)
train_set, test_set = random_split(data, [n_train, n_test], generator=g)
print(len(train_set), len(test_set))

44 10


In [10]:
from torch.utils.data import DataLoader

# feed photos in batches of 8; shuffle only the training set
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
test_loader = DataLoader(test_set, batch_size=8, shuffle=False)

In [11]:
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
# ResNet18 pretrained on millions of photos
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
# replace the last layer: 2 outputs, cube and pen
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)
print(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 43.3MB/s]


cuda


In [12]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()  # how wrong the guesses are
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # adjusts the weights

for epoch in range(5):  # one epoch = one pass over the training photos
    model.train()
    total_loss, correct = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()      # clear old adjustments
        out = model(xb)            # guess
        loss = criterion(out, yb)  # measure the error
        loss.backward()            # work out the adjustments
        optimizer.step()           # apply them
        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
    print(f"epoch {epoch+1}: loss {total_loss/len(train_set):.3f}, train acc {correct/len(train_set):.2f}")

epoch 1: loss 0.308, train acc 0.93
epoch 2: loss 0.030, train acc 1.00
epoch 3: loss 0.032, train acc 0.98
epoch 4: loss 0.004, train acc 1.00
epoch 5: loss 0.010, train acc 1.00


In [13]:
model.eval()  # evaluation mode
correct = 0
with torch.no_grad():  # no learning here, only measuring
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        correct += (out.argmax(1) == yb).sum().item()
print(f"test acc: {correct}/{len(test_set)} = {correct/len(test_set):.2f}")

test acc: 10/10 = 1.00


In [17]:
import time

def time_inference(m, dev, runs=50):
    m = m.to(dev).eval()
    x = torch.randn(1, 3, 224, 224).to(dev)  # one fake photo; timing doesn't depend on content
    with torch.no_grad():
        for _ in range(10):  # warm-up runs, not timed
            m(x)
        if dev == "cuda":
            torch.cuda.synchronize()  # wait until the GPU has finished
        start = time.perf_counter()
        for _ in range(runs):
            m(x)
        if dev == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()
    return (end - start) / runs * 1000  # milliseconds per photo

cpu_ms = time_inference(model, "cpu")
gpu_ms = time_inference(model, "cuda")
print(f"CPU: {cpu_ms:.1f} ms per photo")
print(f"GPU: {gpu_ms:.1f} ms per photo")
print(f"speedup: {cpu_ms/gpu_ms:.1f}x")

CPU: 60.8 ms per photo
GPU: 4.6 ms per photo
speedup: 13.1x


In [15]:
!lscpu | grep -E "Model name|^CPU\(s\)"
!nvidia-smi --query-gpu=name,driver_version --format=csv
print(torch.version.cuda)

CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
name, driver_version
Tesla T4, 580.82.07
13.0
